<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🧪 Evaluating Agents — 把 agent 测明白

**一句话定位**:**3 种粒度 × 2 种方法 + 1 个秘诀** —— 用 LangSmith 把 agent 测明白,不再凭感觉部署。

</div>

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**🤔 为什么这一课是 ambient agent 的「**生死关卡**」**

我们已经搭好了 email assistant。但 —— **你敢真把它接到自己的 Gmail 吗?**

大多数人答案是「**还不敢**」。原因:agent 在生产里会面临 **质量 / 成本 / 安全 / 延迟** 等多重挑战。ambient agent 尤其严重 —— **无人值守**,错一次可能就群发了一堆糟糕邮件。

→ **解药 = 测试**。本节用 **可量化指标**(回信质量、token 消耗、延迟、triage 准确率) 告诉你 agent 表现如何。

</div>

[LangSmith](https://docs.smith.langchain.com/) 提供 **两种主要方式** 测 agent:

![overview-img](img/overview_eval.png)

#### 🔐 加载环境变量

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

## 🛠️ 两种跑评估的方法

### 🅰️ Pytest / Vitest —— 对开发者友好

[Pytest](https://docs.pytest.org/en/stable/) 和 Vitest 是 Python / JavaScript 生态里大家熟悉的测试框架。LangSmith **集成** 了它们,你写普通的 pytest 测试,结果会 **自动记录到 LangSmith**。本课用 Pytest。

| 适用场景 | 原因 |
|---------|------|
| 已经熟悉 pytest 的开发者 | 学习曲线为 **零** |
| 每个 test case 逻辑都不同 | pytest 的 fixture / parametrize **灵活** |
| 本地快速迭代 | 命令行直接跑,IDE 集成好 |

### 🅱️ LangSmith Datasets —— 对团队友好

你也可以在 LangSmith 里 [创建 dataset](https://docs.smith.langchain.com/evaluation),然后用 `client.evaluate()` API 跑测试。

| 适用场景 | 原因 |
|---------|------|
| 团队协作建测试集 | dataset 在 LangSmith **集中管理** |
| 从生产 trace 吸取例子 | annotation queue / 合成数据生成支持 |
| 同一 dataset 跑多种 evaluator | similarity / exact match / LLM judge 可叠加 |

<div class="dark-orange" style="background:#2d2418; color:#fed7aa; padding:10px 24px; border-left:4px solid #fb923c; border-radius:4px; width:97%;"><style>.dark-orange strong{color:#67e8f9;}</style>

💡 **新手选 Pytest,团队用 Datasets**。两种可以共存,看场景挑。

</div>

## 📋 测试用例(Test Cases)

测试常常从 **定义测试用例** 开始,这步并不简单。

本课的测试集在 [`eval/email_dataset.py`](../src/email_assistant/eval/email_dataset.py),包含 4 类信息:

| # | 字段 | 内容 |
|---|------|------|
| 1 | **Input Emails** | 多样化的邮件示例 |
| 2 | **Ground Truth Classifications** | 真值分类:`Respond` / `Notify` / `Ignore` |
| 3 | **Expected Tool Calls** | 对每封需回的邮件,期望调用的工具列表 |
| 4 | **Response Criteria** | 好回信的标准(LLM-as-Judge 用) |

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**🎯 两种测试 = 两种粒度**

- **端到端集成测试**:输入邮件 → agent → **最终输出** vs Response Criteria(LLM 评判)
- **流程单步测试**:输入邮件 → agent → **某一步**(如分类决策)vs Ground Truth(精确匹配)

</div>

In [2]:

%load_ext autoreload
%autoreload 2

import sys
sys.path.append("../src")
from email_assistant.eval.email_dataset import email_inputs, expected_tool_calls, triage_outputs_list, response_criteria_list

test_case_ix = 0

print("Email Input:", email_inputs[test_case_ix])
print("Expected Triage Output:", triage_outputs_list[test_case_ix])
print("Expected Tool Calls:", expected_tool_calls[test_case_ix])
print("Response Criteria:", response_criteria_list[test_case_ix])

Email Input: {'author': 'Alice Smith <alice.smith@company.com>', 'to': 'Lance Martin <lance@company.com>', 'subject': 'Quick question about API documentation', 'email_thread': "Hi Lance,\n\nI was reviewing the API documentation for the new authentication service and noticed a few endpoints seem to be missing from the specs. Could you help clarify if this was intentional or if we should update the docs?\n\nSpecifically, I'm looking at:\n- /auth/refresh\n- /auth/validate\n\nThanks!\nAlice"}
Expected Triage Output: respond
Expected Tool Calls: ['write_email', 'done']
Response Criteria: 
• Send email with write_email tool call to acknowledge the question and confirm it will be investigated  



## 📝 Pytest 示例 — 测 tool 调用

看一下怎么用 Pytest 测某一部分逻辑。这里我们测:**email_assistant 处理邮件时是否调了正确的工具**。

In [3]:
import sys
sys.path.append("../src")

import pytest
from email_assistant.eval.email_dataset import email_inputs, expected_tool_calls
from email_assistant.utils import format_messages_string
from email_assistant.email_assistant import email_assistant
from email_assistant.utils import extract_tool_calls

from langsmith import testing as t

@pytest.mark.langsmith
@pytest.mark.parametrize(
    "email_input, expected_calls",
    [   # Pick some examples with e-mail reply expected
        (email_inputs[0],expected_tool_calls[0]),
        (email_inputs[3],expected_tool_calls[3]),
    ],
)
def test_email_dataset_tool_calls(email_input, expected_calls):
    """Test if email processing contains expected tool calls.
    
    This test confirms that all expected tools are called during email processing,
    but does not check the order of tool invocations or the number of invocations
    per tool. Additional checks for these aspects could be added if desired.
    """
    # Run the email assistant
    messages = [{"role": "user", "content": str(email_input)}]
    result = email_assistant.invoke({"messages": messages})
            
    # Extract tool calls from messages list
    extracted_tool_calls = extract_tool_calls(result['messages'])
            
    # Check if all expected tool calls are in the extracted ones
    missing_calls = [call for call in expected_calls if call.lower() not in extracted_tool_calls]
    
    t.log_outputs({
                "missing_calls": missing_calls,
                "extracted_tool_calls": extracted_tool_calls,
                "response": format_messages_string(result['messages'])
            })

    # Test passes if no expected calls are missing
    assert len(missing_calls) == 0

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**🔑 Pytest 写 LangSmith 测试的 2 个关键点**

1. **`@pytest.mark.langsmith` 装饰器** —— 加上这个,[结果就自动记录到 LangSmith](https://docs.smith.langchain.com/evaluation/how_to_guides/pytest)。把代码放到 `notebooks/test_tools.py` 里就能跑。
2. **`@pytest.mark.parametrize`** —— [传 dataset 示例](https://docs.smith.langchain.com/evaluation/how_to_guides/pytest#parametrize-with-pytestmarkparametrize) 给测试函数,一次跑多个用例。

</div>

### ▶️ 运行 Pytest

从项目根目录跑:

```bash
LANGSMITH_TEST_SUITE='Email assistant: Test Tools For Interrupt' \
    pytest notebooks/test_tools.py
```

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🚨 `LANGSMITH_TEST_SUITE` 环境变量是关键**

不设这个,结果会被记到 LangSmith 但没归类,你在 UI 里找不到。**永远显式设置**。

</div>

### 📊 查看实验结果

在 LangSmith UI 里能看到:

| 列 | 内容来源 |
|----|---------|
| `Pass` 列 | `assert len(missing_calls) == 0` 的结果 |
| `Outputs` 列 | `t.log_outputs(...)` 传入的内容 |
| `Inputs` 列 | 函数参数(`email_input`、`expected_calls`) |

每个 `@pytest.mark.parametrize` 传入的元组 = **一行实验记录**,归到你设的 `LANGSMITH_TEST_SUITE` 名字下,在 LangSmith 的 `Datasets & Experiments` 里能找到。

![Test Results](img/test_result.png)

## 📊 LangSmith Datasets 示例

![overview-img](img/eval_detail.png)

来看怎么用 LangSmith **dataset** 跑评估。

- 上面 Pytest 例子测的是 **tool 调用准确性**
- 这次的 dataset 专门测 **triage 步骤**(分类对不对)

### 📦 Dataset 定义

用 LangSmith SDK [创建 dataset](https://docs.smith.langchain.com/evaluation/how_to_guides/manage_datasets_programmatically#create-a-dataset)。下面的代码会用 [`eval/email_dataset.py`](../src/email_assistant/eval/email_dataset.py) 里的测试用例建一个 dataset。

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**💡 习惯成自然**:**只在 dataset 不存在时才创建**(用 `client.has_dataset()` 判断),避免每次跑 notebook 都建新的。

</div>

In [4]:
from langsmith import Client
import sys
sys.path.append("../src")
from email_assistant.eval.email_dataset import examples_triage

# Initialize LangSmith client
client = Client()

# Dataset name
dataset_name = "E-mail Triage Evaluation"

# Create dataset if it doesn't exist
if not client.has_dataset(dataset_name=dataset_name):
    dataset = client.create_dataset(
        dataset_name=dataset_name, 
        description="A dataset of e-mails and their triage decisions."
    )
    # Add examples to the dataset
    client.create_examples(dataset_id=dataset.id, examples=examples_triage)

### 🎯 Target 函数

dataset 的结构是这样的(input + reference output):

```python
examples_triage = [
  {
      "inputs": {"email_input": email_input_1},
      "outputs": {"classification": triage_output_1},   # 注意:这会变成 reference_output
  }, ...
]
```

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🚨 命名陷阱:你在 dataset 里写 `outputs`,但在 evaluator 函数里参数名要写 `reference_outputs`**

这是 LangSmith 的硬约定 —— `client.create_examples` 时用 `outputs` 字段表示「这是真值/参考输出」,但 `client.evaluate` 把这个字段 **传给 evaluator 函数时,参数名必须是 `reference_outputs`**。

写错就会 **静默拉错线** —— 程序不报错,但分数永远是 0。

</div>

In [5]:
print("Dataset Example Input (inputs):", examples_triage[0]['inputs'])

Dataset Example Input (inputs): {'email_input': {'author': 'Alice Smith <alice.smith@company.com>', 'to': 'Lance Martin <lance@company.com>', 'subject': 'Quick question about API documentation', 'email_thread': "Hi Lance,\n\nI was reviewing the API documentation for the new authentication service and noticed a few endpoints seem to be missing from the specs. Could you help clarify if this was intentional or if we should update the docs?\n\nSpecifically, I'm looking at:\n- /auth/refresh\n- /auth/validate\n\nThanks!\nAlice"}}


In [6]:
print("Dataset Example Reference Output (reference_outputs):", examples_triage[0]['outputs'])

Dataset Example Reference Output (reference_outputs): {'classification': 'respond'}


定义一个函数,接受 dataset 的 `inputs`,**调用我们的 email assistant**。

LangSmith [evaluate API](https://docs.smith.langchain.com/evaluation) 会:

1. 自动把 dataset 的 `inputs` 字典 **传给这个函数**
2. 函数 **返回一个 dict**,代表 agent 的输出
3. 这个返回 dict 会被 **传给 evaluator** 做比对

我们只关心 triage 步骤,所以只需要返回 `classification_decision`。

In [7]:
def target_email_assistant(inputs: dict) -> dict:
    """Process an email through the workflow-based email assistant."""
    response = email_assistant.nodes['triage_router'].invoke({"email_input": inputs["email_input"]})
    return {"classification_decision": response.update['classification_decision']}

### 🎯 Evaluator 函数

现在创建一个 **evaluator 函数**。要测什么?

- **Reference outputs**(来自 dataset):`reference_outputs = {"classification": triage_output_1}`
- **Agent outputs**(来自上面的 target):`outputs = {"classification_decision": agent_output_1}`

我们要 **比对两者** —— 一个简单字符串相等检查就够。

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**📌 evaluator 函数的 2 个固定参数名**

| 参数 | 喂的是什么 |
|------|-----------|
| `outputs` | **target 函数返回的** dict(agent 的输出) |
| `reference_outputs` | dataset 里 example 的 `outputs` 字段(真值) |

**改名就坏!** 这是 LangSmith API 的位置约定,不是普通 Python 关键字参数。

</div>

In [8]:
def classification_evaluator(outputs: dict, reference_outputs: dict) -> bool:
    """Check if the answer exactly matches the expected answer."""
    return outputs["classification_decision"].lower() == reference_outputs["classification"].lower()

### 🚀 运行评估

LangSmith 的 `evaluate()` 把所有连线 **自动搞定**:

| 来源 | 自动传给 |
|------|---------|
| dataset 的 `inputs` | → `target(inputs)` |
| `target` 的返回值 | → evaluator 的 `outputs` 参数 |
| dataset 的 `outputs` 字段 | → evaluator 的 `reference_outputs` 参数 |
| evaluator 的返回值 | → LangSmith 实验记录的 score |

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

**🤝 Pytest vs Evaluate API 的相似之处**

Pytest 用 `@pytest.mark.parametrize` 传 dataset 给测试函数;`evaluate()` 用 dataset 名字 + `target/evaluator` 函数。**两种思路本质相通**,只是组织形式不同。

</div>

In [9]:
# Set to true if you want to kick off evaluation
run_expt = True
if run_expt:
    experiment_results_workflow = client.evaluate(
        # Run agent 
        target_email_assistant,
        # Dataset name   
        data=dataset_name,
        # Evaluator
        evaluators=[classification_evaluator],
        # Name of the experiment
        experiment_prefix="E-mail assistant workflow", 
        # Number of concurrent evaluations
        max_concurrency=2, 
    )

View the evaluation results for experiment: 'E-mail assistant workflow-191c1e28' at:
https://smith.langchain.com/o/ec9ce6f2-a592-4818-9d3c-a22e8f1dbe02/datasets/8fc6ebd6-863b-40c6-b807-8b6512d3fbd6/compare?selectedSessions=08bb4b1c-85ca-408c-8e94-fc4af7953a51




0it [00:00, ?it/s]

🔔 Classification: NOTIFY - This email contains important information
🔔 Classification: NOTIFY - This email contains important information
📧 Classification: RESPOND - This email requires a response
🚫 Classification: IGNORE - This email can be safely ignored
🚫 Classification: IGNORE - This email can be safely ignored
📧 Classification: RESPOND - This email requires a response
📧 Classification: RESPOND - This email requires a response
📧 Classification: RESPOND - This email requires a response
🔔 Classification: NOTIFY - This email contains important information
🚫 Classification: IGNORE - This email can be safely ignored
📧 Classification: RESPOND - This email requires a response
📧 Classification: RESPOND - This email requires a response
🔔 Classification: NOTIFY - This email contains important information
📧 Classification: RESPOND - This email requires a response
🔔 Classification: NOTIFY - This email contains important information
📧 Classification: RESPOND - This email requires a response


两个实验都能在 LangSmith UI 里查看。

![Test Results](img/eval.png)

## 🧠 LLM-as-Judge 评估

我们已经看了 **triage 步骤的单元测试**(用 `evaluate()`)和 **tool 调用测试**(用 Pytest)。

接下来:用 **LLM 当裁判**,根据「成功标准」(success criteria)评估 agent 表现。

![types](img/eval_types.png)

### 🎯 关键技巧:用 structured output 给 grader 一个 schema

先给 LLM grader 定义一个 **structured output schema**,包含 grade(分数)和 justification(理由)两个字段。

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🔑 为什么必须用 `with_structured_output`?**

不用结构化输出,LLM 会回 **裸文本**:`"我觉得满足标准,因为..."`。你得用正则解析、容易 brittle。

用 `with_structured_output(CriteriaGrade)`,LLM 强制返回 Python 对象:

- `result.grade: bool` —— 直接拿来当 True/False
- `result.justification: str` —— 直接看为什么

**裸文本 grader 是 LLM-as-Judge 失败的头号原因**。

</div>

In [29]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

class CriteriaGrade(BaseModel):
    """Score the response against specific criteria."""
    justification: str = Field(description="The justification for the grade and score, including specific examples from the response.")
    grade: bool = Field(description="Does the response meet the provided criteria?")
    
# Create a global LLM for evaluation to avoid recreating it for each test
criteria_eval_llm = init_chat_model("openai:gpt-4o")
criteria_eval_structured_llm = criteria_eval_llm.with_structured_output(CriteriaGrade)

In [ ]:
email_input = email_inputs[0]
print("Email Input:", email_input)
success_criteria = response_criteria_list[0]
print("Success Criteria:", success_criteria)

Email Assistant 用 email 输入跑一遍,把整个消息流格式化成字符串。然后这些都传给 LLM grader,得到一个带 justification 的评分。

In [ ]:
response = email_assistant.invoke({"email_input": email_input})

In [ ]:
from email_assistant.eval.prompts import RESPONSE_CRITERIA_SYSTEM_PROMPT

all_messages_str = format_messages_string(response['messages'])
eval_result = criteria_eval_structured_llm.invoke([
        {"role": "system",
            "content": RESPONSE_CRITERIA_SYSTEM_PROMPT},
        {"role": "user",
            "content": f"""\n\n Response criteria: {success_criteria} \n\n Assistant's response: \n\n {all_messages_str} \n\n Evaluate whether the assistant's response meets the criteria and provide justification for your evaluation."""}
    ])

eval_result

In [ ]:
RESPONSE_CRITERIA_SYSTEM_PROMPT

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**✅ 验证**:LLM grader 返回的结果完美匹配 `CriteriaGrade` 这个 BaseModel 的 schema —— `grade: bool` + `justification: str`。

</div>

## 🧪 大规模测试套件

我们已经看了用 Pytest 和 `evaluate()` 评估 agent,以及 LLM-as-Judge 怎么用。

下一步:**在更大的测试集上跑** —— 这样能更全面地了解 agent 在各种邮件场景的表现。

跑一个大测试集:

```bash
LANGSMITH_TEST_SUITE='Email assistant: Test Full Response Interrupt' \
LANGSMITH_EXPERIMENT='email_assistant' \
    pytest tests/test_response.py --agent-module email_assistant
```

### 📂 `test_response.py` 里你能看到几样东西

**① 用 `@pytest.mark.langsmith` + parametrize 喂数据集**:

```python
@pytest.mark.langsmith(output_keys=["criteria"])
@pytest.mark.parametrize(
    "email_input,email_name,criteria,expected_calls",
    create_response_test_cases()
)
def test_response_criteria_evaluation(email_input, email_name, criteria, expected_calls):
```

**② LLM-as-Judge 的 grading schema**:

```python
class CriteriaGrade(BaseModel):
    """Score the response against specific criteria."""
    grade: bool = Field(description="Does the response meet the provided criteria?")
    justification: str = Field(description="Justification with specific examples from the response.")
```

**③ 用 criteria 跑评估**:

```python
eval_result = criteria_eval_structured_llm.invoke([
    {"role": "system",  "content": RESPONSE_CRITERIA_SYSTEM_PROMPT},
    {"role": "user",
     "content": f"""Response criteria: {criteria}\n\n
                 Assistant's response: {all_messages_str}\n\n
                 Evaluate whether the assistant's response meets the criteria."""}
])
```

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**🎤 Lance 的告诫(听话照做)**

LLM-as-Judge 要花 **1–2 小时迭代**,主要调两件事:

1. **Grader system prompt**:明确「criteria 每条都要满足才 pass」、「**引用具体文本**」
2. **每条 criteria 措辞**:
   - ❌ 太严(「必须用 'Hi Alice' 开头」)→ 全失败
   - ❌ 太松(「调用了 write_email」)→ 全通过,没意义
   - ✅ 刚好(「调用 write_email 表示已收到问题并将调查」)

**调好的标准**:**同一封回信跑 5 次,grade 都一致**(稳定性 > 严格性)。

</div>

### 📊 查看 / 获取结果

在 LangSmith UI 里能看到 agent 哪些做得好、哪些需要改进。

你也可以用 **代码读出实验结果**(适合做自定义可视化):

```python
client = Client()
project = client.read_project(project_name="your_experiment", include_stats=True)
print("Latency p50:", project.latency_p50)
print("Latency p99:", project.latency_p99)
print("Token Usage:", project.total_tokens)
print("Feedback Stats:", project.feedback_stats)
```

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

## ✨ 本节带走

**Evaluate 的核心 = 3 粒度 × 2 方法 + 1 秘诀**:

- **3 粒度**:triage 决策(粗)→ tool calls(中)→ 整体回信(细)
- **2 方法**:Pytest(开发友好)/ Evaluate API(团队友好)
- **1 秘诀**:LLM-as-Judge 必须用 `with_structured_output` —— `grade + justification` 是稳定性的根

🎯 **3 句工程口诀**:

1. **能用启发式比较就别上 LLM-as-Judge** —— 字符串 == 又快又稳又免费
2. **评估代码要进 git** —— 每次改 agent 跑一遍,回归测试是 agent 的硬通货
3. **golden dataset 持续养大** —— 把生产 trace 中的好/坏例子都吸进 dataset

📎 配套精华笔记:[`2_z_⭐️_本节精华_Evaluate.ipynb`](./2_z_⭐️_本节精华_Evaluate.ipynb)

</div>

In [34]:
# TODO: Copy your experiment name here
experiment_name = "email_assistant:8286b3b8"
# Set this to load expt results
load_expt = False
if load_expt:
    email_assistant_experiment_results = client.read_project(project_name=experiment_name, include_stats=True)
    print("Latency p50:", email_assistant_experiment_results.latency_p50)
    print("Latency p99:", email_assistant_experiment_results.latency_p99)
    print("Token Usage:", email_assistant_experiment_results.total_tokens)
    print("Feedback Stats:", email_assistant_experiment_results.feedback_stats)